In [1]:
import os

notebook_dir = "/home/balabaevvl/courses/sdc/ysda_sdc/seminar08-ml-planning/homework"
# notebook_dir = "/root/all/study/tmp_ysda_sdc/ysda_sdc/seminar08-ml-planning/homework"

os.chdir(notebook_dir)

GPUs = [
    "GPU-e83bd31b-fcb9-b8de-f617-2d717619413b",
    "GPU-5a9b7750-9f85-49a5-3aae-fe07b1b7661d",
    "GPU-fe2d8dfd-06f2-a5c4-a7fd-4a5f23947005",
    "GPU-0c320096-21ee-4060-8731-826ca2febfab",
    "GPU-baef952c-6609-aace-3b78-e4e07788d5de",
    "GPU-3979d65b-c238-4e9c-0c1c-1aa3f05c56a1",
    "GPU-6c76a2c5-5375-aa06-11d4-0fddfac30e91",
]
os.environ["CUDA_VISIBLE_DEVICES"] = f"{GPUs[2]}"


In [2]:
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"


# Scenario Data Loading

This tutorial demonstrates how to load scenario data from the Waymo Open Motion Dataset (WOMD) using the Waymax dataloader.

In [3]:
# !pip install --upgrade pip
# !pip install git+https://github.com/waymo-research/waymax.git@main#egg=waymo-waymax
# !pip install numpy==2.3.5
# !pip install matplotlib

Семпл данных для семинара лежит тут https://disk.yandex.ru/d/IoFBUM-OHDKh4w

**Обратите внимание, архив отличается от архива с семинара наличием тестов и размером датасета**

In [4]:
# !rm -f data.tar.gz
# !wget -O data.tar.gz "$(curl -s 'https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key=https://disk.yandex.ru/d/IoFBUM-OHDKh4w' | python3 -c 'import sys, json; print(json.load(sys.stdin)["href"])')"
# !mkdir -p ysda-planning-rl
# !tar -xf data.tar.gz -C ysda-planning-rl

In [5]:
SEMINAR_PATH = 'ysda-planning-rl'

In [6]:
import os
import shutil

if not os.path.exists('lib'):
    shutil.copytree(os.path.join(SEMINAR_PATH, 'lib'), 'lib')
else:
    print('"lib" folder already exists. If you want to rewrite lib by original folder, remove local "lib" manually')

"lib" folder already exists. If you want to rewrite lib by original folder, remove local "lib" manually


In [7]:
import os
import dataclasses as dc
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


from waymax.config import DatasetConfig

from lib.data_utils import WaymaxDataset, scenario_to_features_gt

device = 'cuda'


class PlannerModel(nn.Module):
    def __init__(
        self,
        n_modes=6,
        future_steps=30,
        hidden_dim=128,
        history_size=11,
        n_heads=4,
        n_history_layers=1,
        n_encoder_layers=3,
        n_decoder_layers=2,
        dropout=0.1,
    ):
        super().__init__()
        self.n_modes = n_modes
        self.future_steps = future_steps
        self.hidden_dim = hidden_dim
        self.history_size = history_size

        agent_step_dim = 9
        self.agent_step_proj = nn.Sequential(
            nn.Linear(agent_step_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.agent_time_embed = nn.Parameter(torch.randn(history_size, hidden_dim) * 0.02)
        self.history_cls = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)
        history_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_heads, dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu',
        )
        self.history_encoder = nn.TransformerEncoder(history_layer, num_layers=n_history_layers)

        self.type_embed = nn.Embedding(8, 16)
        self.agent_static_proj = nn.Sequential(
            nn.Linear(16 + 1, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        self.agent_fuse = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.map_type_embed = nn.Embedding(22, 16)
        self.map_proj = nn.Sequential(
            nn.Linear(4 + 16, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.agent_token_type = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)
        self.map_token_type = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)

        scene_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_heads, dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu',
        )
        self.scene_encoder = nn.TransformerEncoder(scene_layer, num_layers=n_encoder_layers)

        self.mode_queries = nn.Parameter(torch.randn(n_modes, hidden_dim) * 0.02)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim, nhead=n_heads, dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu',
        )
        self.mode_decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_decoder_layers)

        self.traj_head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, future_steps * 6),
        )
        self.logit_head = nn.Sequential(
            nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, 1),
        )

    @staticmethod
    def _to_local(x, y, ex, ey, cos_y, sin_y):
        dx = x - ex
        dy = y - ey
        return dx * cos_y + dy * sin_y, -dx * sin_y + dy * cos_y

    @staticmethod
    def _rotate(vx, vy, cos_y, sin_y):
        return vx * cos_y + vy * sin_y, -vx * sin_y + vy * cos_y

    def forward(self, features):
        device_ = features['log_trajectory']['x'].device
        bs, n_a, n_t = features['log_trajectory']['x'].shape
        K, H = self.n_modes, self.hidden_dim

        agent_mask = features['agent_to_predict_mask']
        bidx = torch.arange(bs, device=device_)
        pred_idx = agent_mask.float().argmax(dim=-1)

        ex = features['log_trajectory']['x'][bidx, pred_idx, -1]
        ey = features['log_trajectory']['y'][bidx, pred_idx, -1]
        eyaw = features['log_trajectory']['yaw'][bidx, pred_idx, -1]
        cos_y, sin_y = torch.cos(eyaw), torch.sin(eyaw)

        traj = features['log_trajectory']
        valid = traj['valid'].float()
        rx, ry = self._to_local(
            traj['x'], traj['y'],
            ex[:, None, None], ey[:, None, None],
            cos_y[:, None, None], sin_y[:, None, None],
        )
        rvx, rvy = self._rotate(
            traj['vel_x'], traj['vel_y'],
            cos_y[:, None, None], sin_y[:, None, None],
        )
        ryaw = traj['yaw'] - eyaw[:, None, None]

        v3 = valid
        per_step = torch.stack([
            rx * v3, ry * v3, rvx * v3, rvy * v3,
            torch.sin(ryaw) * v3, torch.cos(ryaw) * v3,
            v3, traj['length'] * v3, traj['width'] * v3,
        ], dim=-1)
        per_step = self.agent_step_proj(per_step) + self.agent_time_embed[None, None]
        per_step = per_step.reshape(bs * n_a, n_t, H)
        cls = self.history_cls.expand(bs * n_a, -1, -1)
        seq = torch.cat([cls, per_step], dim=1)
        time_pad = valid.reshape(bs * n_a, n_t) < 0.5
        cls_pad = torch.zeros(bs * n_a, 1, dtype=torch.bool, device=device_)
        seq_pad = torch.cat([cls_pad, time_pad], dim=1)
        seq_enc = self.history_encoder(seq, src_key_padding_mask=seq_pad)
        agent_hist = seq_enc[:, 0].reshape(bs, n_a, H)

        types = features['object_metadata']['object_types']
        is_ego = features['object_metadata']['is_sdc'].float().unsqueeze(-1)
        type_emb = self.type_embed((types.clamp(min=-1) + 1).long())
        static = self.agent_static_proj(torch.cat([type_emb, is_ego], dim=-1))
        agent_tokens = self.agent_fuse(torch.cat([agent_hist, static], dim=-1)) + self.agent_token_type

        ids = features['object_metadata']['ids']
        agent_valid = (ids >= 0) & (valid.sum(-1) > 0)

        rg = features['roadgraph_points']
        rg_valid = rg['valid'].float()
        rg_rx, rg_ry = self._to_local(
            rg['x'], rg['y'], ex[:, None], ey[:, None],
            cos_y[:, None], sin_y[:, None],
        )
        rg_rdx, rg_rdy = self._rotate(
            rg['dir_x'], rg['dir_y'], cos_y[:, None], sin_y[:, None],
        )
        rg_rx = rg_rx * rg_valid; rg_ry = rg_ry * rg_valid
        rg_rdx = rg_rdx * rg_valid; rg_rdy = rg_rdy * rg_valid
        rg_type_emb = self.map_type_embed((rg['types'].clamp(min=-1) + 1).long())
        rg_feats = torch.cat([
            rg_rx.unsqueeze(-1), rg_ry.unsqueeze(-1),
            rg_rdx.unsqueeze(-1), rg_rdy.unsqueeze(-1),
            rg_type_emb,
        ], dim=-1)
        map_tokens = self.map_proj(rg_feats) + self.map_token_type
        map_valid = rg_valid > 0.5

        scene = torch.cat([agent_tokens, map_tokens], dim=1)
        scene_valid = torch.cat([agent_valid, map_valid], dim=1)
        scene_pad = ~scene_valid
        scene_enc = self.scene_encoder(scene, src_key_padding_mask=scene_pad)

        mode_q = self.mode_queries.unsqueeze(0).expand(bs, -1, -1)
        mode_out = self.mode_decoder(
            tgt=mode_q, memory=scene_enc, memory_key_padding_mask=scene_pad,
        )
        traj_local = self.traj_head(mode_out).reshape(bs, K, self.future_steps, 6)
        logits = self.logit_head(mode_out).squeeze(-1)

        loc_x, loc_y = traj_local[..., 0], traj_local[..., 1]
        loc_vx, loc_vy = traj_local[..., 2], traj_local[..., 3]
        loc_yaw = torch.atan2(traj_local[..., 4], traj_local[..., 5])

        cos_g, sin_g = cos_y[:, None, None], sin_y[:, None, None]
        glb_x = loc_x * cos_g - loc_y * sin_g + ex[:, None, None]
        glb_y = loc_x * sin_g + loc_y * cos_g + ey[:, None, None]
        glb_vx = loc_vx * cos_g - loc_vy * sin_g
        glb_vy = loc_vx * sin_g + loc_vy * cos_g
        glb_yaw = loc_yaw + eyaw[:, None, None]

        return {
            'trajectory': {
                'x': glb_x, 'y': glb_y, 'yaw': glb_yaw,
                'vel_x': glb_vx, 'vel_y': glb_vy,
            },
            'logits': logits,
        }


PlanningModel = PlannerModel


def _features_to_device(obj, device):
    if isinstance(obj, torch.Tensor):
        return obj.to(device, non_blocking=True)
    if isinstance(obj, dict):
        return {k: _features_to_device(v, device) for k, v in obj.items()}
    if dc.is_dataclass(obj):
        return dc.replace(obj, **{
            f.name: _features_to_device(getattr(obj, f.name), device)
            for f in dc.fields(obj)
        })
    return obj


class NormalizeSceneWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, features):
        device_ = next(self.model.parameters()).device
        return self.model(_features_to_device(features, device_))

## Здесь вам нужно загрузить веса своей обученной модели планера из предыдущего дз

In [8]:
checkpoint_path = '../../seminar07-ml-planning/homework/checkpoints/planner-08-0.630.ckpt'

checkpoint = torch.load(checkpoint_path, map_location='cpu')

if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    full_state = checkpoint['state_dict']
    inner_state_dict = {
        k[len('model.'):]: v
        for k, v in full_state.items()
        if k.startswith('model.')
    }
else:
    inner_state_dict = checkpoint

planner = PlannerModel()
missing, unexpected = planner.load_state_dict(inner_state_dict, strict=False)
print(f'Loaded planner. missing={len(missing)}, unexpected={len(unexpected)}')

model = NormalizeSceneWrapper(planner).to(device)
model.eval()

/home/balabaevvl/courses/venv311/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loaded planner. missing=0, unexpected=0


NormalizeSceneWrapper(
  (model): PlannerModel(
    (agent_step_proj): Sequential(
      (0): Linear(in_features=9, out_features=128, bias=True)
      (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
    )
    (history_encoder): TransformerEncoder(
      (layers): ModuleList(
        (0): TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (type

In [9]:
def get_data_config(split_name, seminar_path=SEMINAR_PATH):
    split_path = os.path.join(SEMINAR_PATH, 'data', split_name)

    obj_count = int(os.listdir(split_path)[0].rsplit('-')[-1])
    return DatasetConfig(
        path=os.path.join(split_path, f'{split_name}_tfexample.tfrecord@{obj_count}'),
        max_num_objects=12,
        batch_dims=[4],
        repeat=1,
        shuffle_buffer_size=64,
        deterministic=False,
        num_shards=1
    )

In [10]:
train_dataset = WaymaxDataset(get_data_config('training'))
val_dataset = WaymaxDataset(get_data_config('validation'))

In [11]:
FUTURE_STEPS = 30

# Дообучение модели планнера с помощью RL

В данном домашнем задании вам предстоит реализовать алгоритм PPO - Proximal Policy Optimization для дообучения модели планнинга. Для того чтобы ознакомиться с идеей алгоритма, рекомендую прочесть полную статью https://arxiv.org/pdf/1707.06347 или краткую сводку с всей нужной информацией https://spinningup.openai.com/en/latest/algorithms/ppo.html

In [12]:
from waymax import config as _config
from waymax import dataloader
from waymax import datatypes
from waymax import dynamics
from waymax import env as _env
from waymax import agents
from waymax import visualization
from waymax.agents import SimAgentActor, WaymaxActorOutput
from waymax import metrics

from torch.distributions import Categorical

import jax
from jax import numpy as jnp
from jax import random

import chex

import dataclasses

from typing import Any, Optional, Callable, Sequence
import mediapy

# PPO Algorithm

## Initializing the ego-agent as RL agent

**Класс EgoAgent** является классом, который управляет эго-агентом в симуляции. Он наследуется от класса SimAgentActor из Waymax и расширяет его функциональность для работы с моделью планнера, которая предсказывает следующие действия агента.


1. Имплементация метода select_action не отличается от базовой имплементации, кроме того, что мы добавляем поле log_probs в выход, чтобы во время обучения использовать их для подсчета ppo loss.


2. Метод update_trajectory является ключевым в классе EgoAgent. Он отвечает за выбор действия на основе текущей политики и возвращает логарифмические вероятности действий, которые используются для вычисления лосса в алгоритме PPO.

In [13]:
def extract_best_mode_from_pred_component(pred, best_mode):
    return pred[torch.arange(best_mode.shape[0], device=best_mode.device), best_mode]

def extract_first_timestep_from_pred(pred):
    return pred[..., 0]

In [14]:
ActorState = datatypes.PyTree
Params = datatypes.PyTree
Action = datatypes.PyTree


@chex.dataclass(frozen=True)
class RLActorOutput(agents.WaymaxActorOutput):
    """Output of the RL actor, extending WaymaxActorOutput with log_probs.

    Attributes:
        actor_state: Internal state for whatever the agent needs to keep as its
          state. This can be recurrent embeddings or accounting information.
        action: Action of shape (..., num_objects) predicted by the RL actor.
        is_controlled: A binary indicator of shape (..., num_objects) representing
          which objects are controlled by the actor.
        log_probs: Log_probs output by the RL actor, representing the log probability
          of policy's actions
    """

    log_probs: Optional[jax.Array]


class EgoAgent(SimAgentActor):
    def __init__(self,  model, is_controlled_func: Optional[Callable[[datatypes.SimulatorState], jax.Array]] = None):
        super().__init__(is_controlled_func=is_controlled_func)
        self.model = model


    def update_trajectory(
        self, state: datatypes.SimulatorState
    ) -> datatypes.TrajectoryUpdate:
        """Updates the trajectory for all simulated agents."""

        features, _ = scenario_to_features_gt(state, features_first_timestamp=state.timestep - 10, gt_timestamps=1)
        pred = self.model(features)
        logits = pred['logits']
        mode_distribution = Categorical(logits=logits)
        mode_sample = mode_distribution.sample()
        log_probs = mode_distribution.log_prob(mode_sample)

        trajectory_jax = jax.tree_util.tree_map(
            lambda x: jnp.repeat(
                jnp.array(
                    extract_first_timestep_from_pred(
                        extract_best_mode_from_pred_component(x, mode_sample)
                    ).cpu().detach().numpy()
                ).reshape(-1, 1, 1),
                state.log_trajectory.x.shape[-2],
                axis=-2
            ),
            pred['trajectory']
        )

        x = trajectory_jax['x']
        y = trajectory_jax['y']
        vel_x = trajectory_jax['vel_x']
        vel_y = trajectory_jax['vel_y']
        yaw = trajectory_jax['yaw']

        # predictions is valid only for sdc
        valid = jnp.bool_(jnp.zeros_like(x))
        valid = valid.at[state.object_metadata.is_sdc].set(True)

        return datatypes.TrajectoryUpdate(x=x, y=y, yaw=yaw, vel_x=vel_x, vel_y=vel_y, valid=valid), log_probs


    def select_action(
        self,
        params: Params,
        state: datatypes.SimulatorState,
        actor_state: Any,
        rng: jax.Array,
    ) -> agents.WaymaxActorOutput:
        """Selects action and updates trajectory given the current simulator state."""

        del actor_state, rng  # Not used
        action, log_probs = self.update_trajectory(state)
        action = action.as_action() # here we transform the action which we got from the model to a desired datatype [look above]

        return RLActorOutput(
            action=action,
            actor_state=None,
            is_controlled=self.is_controlled_func(state),
            log_probs=log_probs,
        )

    @property
    def name(self) -> str:
        return self.__class__.__name__

## Rewards, Reward-to-go computation and PPO Loss function

### Rewards {1.5 балла}

Ниже вам нужно будет создать различные функции для подсчета наград при помощи модуля waymax.rewards. Эти функции наград используются в симуляции для оценки поведения агентов в зависимости от их действий. Зайдите в репозиторий https://github.com/waymo-research/waymax/tree/main/waymax/metrics и поймите, как добавлять следующие реворды в обучение: imitation reward, offroad reward, overlap reward, comfort reward. Также, разберитесь в смысле каждого реворда, исходя из кода waymax/metrics, и опишите их здесь:

1. **Log divergence** (`waymax/metrics/imitation.py::LogDivergenceMetric`, имя `log_divergence`): на каждом шаге считает евклидову L2-дистанцию (xy) между текущей позицией SDC в `sim_trajectory` и его же позицией в `log_trajectory` на том же таймстепе. Чем ближе агент к человеческому логу — тем меньше значение, поэтому это **имитационный сигнал**. Метрика возвращает положительное число (расстояние в метрах). Чтобы получить *reward*, который агент максимизирует, в конфиге задаём отрицательный вес — `-1.0`.

2. **Offroad reward** (`waymax/metrics/roadgraph.py::OffroadMetric`, имя `offroad`): индикатор `1.0`, если хотя бы один угол bbox SDC оказывается за пределами дорожной зоны (по road-edge сегментам в roadgraph), и `0.0` иначе. Хотим минимизировать съезды с дороги, поэтому вес тоже **отрицательный** (а сам коэффициент берём побольше — `-3.0`, чтобы политика сильно избегала offroad).

3. **Overlap reward** (`waymax/metrics/overlap.py::OverlapMetric`, имя `overlap`): индикатор `1.0`, если bbox SDC пересекает bbox любого другого валидного агента на сцене (т.е. произошло столкновение), `0.0` иначе. Сталкиваться нельзя — поэтому вес **отрицательный и большой** (`-3.0`).

4. **Comfort reward** (`waymax/metrics/comfort.py::KinematicsInfeasibilityMetric`, имя `kinematic_infeasibility`): индикатор кинематической невыполнимости перехода (по факту — слишком резкое ускорение / угловая скорость, выходящие за реалистичный для машины диапазон). Это служит прокси для «комфорта»: машина не должна дёргаться. Вес **отрицательный**. В нашем основном `combination_reward_function` мы его не включаем по умолчанию (чтобы не перегружать задачу), но он реализован отдельным конфигом, и его можно добавить, если модель начнёт ехать рывками.

Общее правило знака: метрики waymax возвращают «штраф» — большое значение = плохо. Алгоритмы RL максимизируют reward, поэтому веса в `LinearCombinationRewardConfig` ставим со **знаком минус**, чтобы превратить штраф в награду.

С помощью linear combination reward из модуля waymax мы можем совмещать несколько наград и считать суммарную награду на каждом шаге симуляции. Давайте для начала добавим log_divergence, offroad, overlap rewards

В конфиге награды можно менять величину штрафа. Например, если мы видим, что модель склонна часто выезжать за пределы дороги, можно увеличить штраф за offroad, чтобы модель с большей вероятностью отвергала действия, приводящие к выезду за пределы дороги.

Также, посмотрев в код наград, подумайте, с каким знаком их нужно добавлять в наше обучение. Здесь важно помнить, что алгоритмы RL максимизируют награду

In [15]:
from waymax.rewards import linear_combination_reward

imitation_config = _config.LinearCombinationRewardConfig({'log_divergence': -1.0})
imitation_reward_function = linear_combination_reward.LinearCombinationReward(imitation_config)

offroad_config = _config.LinearCombinationRewardConfig({'offroad': -1.0})
offroad_reward_function = linear_combination_reward.LinearCombinationReward(offroad_config)

overlap_config = _config.LinearCombinationRewardConfig({'overlap': -1.0})
overlap_reward_function = linear_combination_reward.LinearCombinationReward(overlap_config)

all_rewards_config = _config.LinearCombinationRewardConfig({
    'log_divergence': -1.0,
    'offroad': -3.0,
    'overlap': -3.0,
})
combination_reward_function = linear_combination_reward.LinearCombinationReward(all_rewards_config)

#### Задания 1.1 и 1.2 {2 балла}:

1.1 **Имплементировать PPO-Clip Loss** {1.5 баллов}
PPO-Clip Loss используется для оптимизации политики агента. PPO-Clip обновляет веса с помощью градиентного подъема:
$$
\theta_{k+1} = \arg \max_{\theta} \mathbb{E}_{s,a \sim \pi_{\theta_k}} \left[ L(s, a, \theta_k, \theta) \right],
$$
L имеет вид:
$$
L(s, a, \theta_k, \theta) = \min\left(
    \frac{\pi_\theta(a|s)}{\pi_{\theta_k}(a|s)} A^{\pi_{\theta_k}}(s, a), \;
    \text{clip}\left(
        \frac{\pi_\theta(a|s)}{\pi_{\theta_k}(a|s)}, 1 - \epsilon, 1 + \epsilon
    \right) A^{\pi_{\theta_k}}(s, a)
\right),
$$

Epsilon - это clipping factor, контролирующий, насколько далеко от референсной политики $\pi_{\theta_k}$ может уйти обучаемая политика $\pi_\theta$.
Так как современные оптимайзеры решают задачу минимизации, а в PPO мы наоборот максимизируем ожидаемую награду, то при обучении будем домножать лосс на -1, чтобы градиентный спуск фактически поднимал reward.

В стандартном случае, $A(s, a) = Q(s, a) - V(s)$, где $V(s)$ - произвольная функция или модель, которая умеет оценивать бейзлайны, то есть среднюю награду, которую получит наш агент, начав в стейте s и придерживаясь политики $\pi_\theta$ до конца эпизода. Основная цель добавления $V(s)$ в подсчет advantage-ей - снизить дисперсию в оценке реворда. Для того чтобы не нагромождать эту дз, мы не будем обучать отдельную Value Model для оценки бейзлайнов, вместо этого будем считать, что $A(s, a) = normalized (Q(s, a))$, что так же позволяет снизить дисперсию.

1.2  **Имплементировать Reward-to-go** {0.5 балл}


$Q(s, a)$ - функция, оценивающая ожидаемый куммулятивный реворд от того, что агент принял action a в стейте s. Есть различные способы оценивать $Q(s, a)$, например - мгновенная награда от выполнения action a или сумма всех наград за эпизод. Чтобы оценить полезность action a в state s, агент должен ориентироваться на последствия от предпринятого action-a. Реворды, которые были получены до этого момента не показывают, насколько хорошее был действие. Соответственно, мы хотим считать $Q(s, a)$ как куммулятивную сумму будущих ревордов (reward-to-go)


Формула для Reward-to-Go:
$$ G_t = \sum_{t'=t}^{T} \gamma^{t'-t} \cdot r_{t'} $$
Где: $ r_{t'} $ — награда на шаге t′. $γ$ — коэффициент дисконтирования.



In [16]:
def compute_PPO_loss(advantages, ratio, clip_coef=0.1):
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - clip_coef, 1.0 + clip_coef) * advantages
    loss_per_step = -torch.min(surr1, surr2)
    return loss_per_step.mean(dim=-1)


def reward2go(reward, gamma=0.99):
    bs, T = reward.shape
    rtgs = torch.zeros_like(reward)
    running = torch.zeros(bs, device=reward.device, dtype=reward.dtype)
    for t in range(T - 1, -1, -1):
        running = reward[:, t] + gamma * running
        rtgs[:, t] = running
    return rtgs

## Initializing the environment and actors around us

#### Задание 2.1 {1 балл}:
В объекте state.object_metadata содержится информация о всех агентах на сцене.
**Найдите в этом объекте маску, которая возвращает True для агентов, соответствующих беспилотному автомобилю (SDC).**

**Настройте управление агентами:**
Для актора IDM_actors настройте функцию is_controlled_func так, чтобы она возвращала True для всех агентов, которые не являются беспилотным автомобилем.
Для актора actor_ego настройте функцию is_controlled_func так, чтобы она возвращала True только для агентов, которые являются беспилотным автомобилем.

In [17]:
dynamics_model = dynamics.StateDynamics()

# Number of agents on the scene, can be changed if you have enough compute to take actions for more agents in your env
max_num_objects = 12

env = _env.BaseEnvironment(
    dynamics_model=dynamics_model,
    config=dataclasses.replace(
        _config.EnvironmentConfig(),
        max_num_objects=max_num_objects,
        controlled_object=_config.ObjectType.VALID,
    ),
)

In [18]:
IDM_actors = agents.IDMRoutePolicy(
    is_controlled_func=lambda state: state.object_metadata.is_sdc == False
)

actor_ego = EgoAgent(
    model,
    is_controlled_func=lambda state: state.object_metadata.is_sdc == True,
)

actors = [actor_ego, IDM_actors]

select_action_list = [actor.select_action for actor in actors]
step = env.step

# Train loop for PPO {5 баллов}

В данном задании вам предстоит заполнить пропуски в train loop-е для обучения алгоритма PPO.

In [19]:
def train_one_batch_ppo(scenario, start_timestep, gamma=1, clip_epsilon=0.2, ppo_epochs=3):

    rollout_data = {
        'states': [],
        'actions': [],
        'log_probs': [],
        'rewards': [],
        'masks': [],
    }

    current_state = env.reset(scenario)
    for timestep in range(start_timestep, start_timestep + FUTURE_STEPS):

        with torch.no_grad():
            outputs = [
                select_action({'timestep': timestep}, current_state, None, None)
                for select_action in select_action_list
            ]
            action = agents.merge_actions(outputs)
            log_probs = outputs[0].log_probs.to(device)

        agent_mask = current_state.object_metadata.is_sdc

        total_reward = combination_reward_function.compute(current_state, action, agent_mask).mean(axis=-1)
        reward = torch.tensor(np.asarray(total_reward), device=device)

        rollout_data['states'].append(current_state)
        rollout_data['log_probs'].append(log_probs)
        rollout_data['rewards'].append(reward)
        rollout_data['masks'].append(agent_mask)

        next_state = step(current_state, action)
        current_state = next_state


    rewards = torch.stack(rollout_data['rewards'], dim=1)
    old_log_probs = torch.stack(rollout_data['log_probs'], dim=1)

    rtgs = reward2go(rewards, gamma=gamma)
    normalized_rtgs = (rtgs - rtgs.mean()) / (rtgs.std() + 1e-8)

    for _ in range(ppo_epochs):
        new_log_probs = []
        for timestep in range(FUTURE_STEPS):
            outputs = [
                select_action({'timestep': start_timestep + timestep},
                              rollout_data['states'][timestep], None, None)
                for select_action in select_action_list
            ]

            new_log_prob = outputs[0].log_probs.to(device)
            new_log_probs.append(new_log_prob)

        new_log_probs = torch.stack(new_log_probs, dim=1)
        ratios = torch.exp(new_log_probs - old_log_probs)

        policy_loss = compute_PPO_loss(normalized_rtgs, ratios, clip_coef=clip_epsilon).mean()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

    avg_step_reward = torch.stack(rollout_data['rewards']).mean(dim=0).mean().item()

    del rewards, log_probs, rtgs, normalized_rtgs
    torch.cuda.empty_cache()

    return {
        'loss': policy_loss.item(),
        'avg_step_reward': avg_step_reward,
    }

In [20]:
optimizer = torch.optim.Adam(model.parameters(), lr=2.5e-4)
epochs = 1
gamma = 0.99
log_interval = 3
clip_epsilon = 0.2
ppo_epochs = 3

PPO_CKPT_DIR = 'ppo_checkpoints'
os.makedirs(PPO_CKPT_DIR, exist_ok=True)
SAVE_EVERY = 50

best_avg_step_reward = float('-inf')

for epoch in range(epochs):
    for scenario_idx, scenario in enumerate(train_dataset):
        metrics = train_one_batch_ppo(
            scenario,
            start_timestep=11,
            gamma=gamma,
            clip_epsilon=clip_epsilon,
            ppo_epochs=ppo_epochs
        )

        if (scenario_idx + 1) % SAVE_EVERY == 0:
            torch.save(
                {'model': model.state_dict(),
                 'optimizer': optimizer.state_dict(),
                 'epoch': epoch,
                 'scenario_idx': scenario_idx,
                 'metrics': metrics},
                os.path.join(PPO_CKPT_DIR, 'last.pt'),
            )
        if metrics['avg_step_reward'] > best_avg_step_reward:
            best_avg_step_reward = metrics['avg_step_reward']
            torch.save(
                {'model': model.state_dict(),
                 'avg_step_reward': metrics['avg_step_reward'],
                 'epoch': epoch,
                 'scenario_idx': scenario_idx},
                os.path.join(PPO_CKPT_DIR, 'best.pt'),
            )

        if (scenario_idx + 1) % log_interval == 0:
            print(f"Epoch {epoch}, Scenario {scenario_idx + 1}:")
            print(f"  Total Loss: {metrics['loss']:.4f}")
            print(f"  Avg Step Reward: {metrics['avg_step_reward']:.4f}")
            print(f"  Best so far:     {best_avg_step_reward:.4f}")
            print("-" * 40)

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: (<gast.gast.NamedExpr object at 0x7fd5941e21c0>, (leaf_jax_array := getattr(leaf, '__jax_array__', None)))
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: (<gast.gast.NamedExpr object at 0x7fd5941e21c0>, (leaf_jax_array := getattr(leaf, '__jax_array__', None)))
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Cause: could not parse the source code of <function <lambda> at 0x7fd6c3769d30>: no matching AST found among candidates:
# coding=utf-8
lambda self: self._jaxpr
# coding=utf-8
lambda self: self._consts
# coding=utf-8
lambda self: self._jaxpr.constvars
# coding=utf

KeyboardInterrupt: 

### Можете остановить обучение, когда будете достигать Avg Step Reward = -0.15 или когда у вас кончится квота в колабе:)

# Проверим обученную модель на валидационном датасете

Так как делать шаги в симуляторе довольно дорого и долго, мы прогоним на валидации только часть датасета, чтобы не забирать у вас всю гпу квоту. Если у вас заканчивается квота, можете уменьшить количество сценариев

In [21]:
def evaluation(dataset, history_size=11, log_interval=1, num_scenarios=5):
    metrics = {
        'total': [],
        'distance': [],
        'offroad': [],
        'overlap': []
    }

    for i, scenario in enumerate(dataset, 1):
        episode_metrics = {k: [] for k in metrics}
        state = env.reset(scenario)

        for timestep in range(history_size, history_size + FUTURE_STEPS):
            with torch.no_grad():
                outputs = [select_action({'timestep': timestep}, state, None, None)
                         for select_action in select_action_list]
                action = agents.merge_actions(outputs)
                mask = state.object_metadata.is_sdc

            # Compute all rewards at once
            rewards = {
                'total': combination_reward_function.compute(state, action, mask),
                'distance': imitation_reward_function.compute(state, action, mask),
                'offroad': offroad_reward_function.compute(state, action, mask),
                'overlap': overlap_reward_function.compute(state, action, mask)
            }

            # Store rewards
            for k in episode_metrics:
                episode_metrics[k].append(torch.tensor(np.asarray(rewards[k]), device=device).mean(axis=-1))

            state = step(state, action)

        # Stack and store episode results
        for k in metrics:
            metrics[k].append(torch.stack(episode_metrics[k], dim=1))

        # Periodic logging
        if i % log_interval == 0:
            print(f"Validation Metrics, step: {i}")
            for k, v in metrics.items():
                mean_reward = torch.mean(torch.cat([m.mean(dim=1) for m in v]))
                print(f"Mean Episode {k.capitalize()} Reward: {mean_reward.item():.2f}")
        if i == num_scenarios:
            break

evaluation(val_dataset)

Validation Metrics, step: 1
Mean Episode Total Reward: -0.03
Mean Episode Distance Reward: -0.03
Mean Episode Offroad Reward: 0.00
Mean Episode Overlap Reward: 0.00
Validation Metrics, step: 2
Mean Episode Total Reward: -0.03
Mean Episode Distance Reward: -0.03
Mean Episode Offroad Reward: 0.00
Mean Episode Overlap Reward: 0.00
Validation Metrics, step: 3
Mean Episode Total Reward: -0.02
Mean Episode Distance Reward: -0.02
Mean Episode Offroad Reward: 0.00
Mean Episode Overlap Reward: 0.00
Validation Metrics, step: 4
Mean Episode Total Reward: -0.02
Mean Episode Distance Reward: -0.02
Mean Episode Offroad Reward: 0.00
Mean Episode Overlap Reward: 0.00
Validation Metrics, step: 5
Mean Episode Total Reward: -0.02
Mean Episode Distance Reward: -0.02
Mean Episode Offroad Reward: 0.00
Mean Episode Overlap Reward: 0.00


# Rollout generation - генерация видосиков {0.5 балла}
Скачайте несколько роллаутов и приложите их вместе с домашним заданием

In [22]:
def generate_close_loop(model, scenario, steps=None):
    dynamics_model = dynamics.StateDynamics()

    max_num_objects = 12

    # Environment to control all objects on scene
    env = _env.BaseEnvironment(
        dynamics_model=dynamics_model,
        config=dataclasses.replace(
            _config.EnvironmentConfig(),
            max_num_objects=max_num_objects,
            controlled_object=_config.ObjectType.VALID,
        ),
    )

    state = env.reset(scenario)

    # intelligent driver model actor for non-ego objects
    IDM_actors = agents.IDMRoutePolicy(
        is_controlled_func=lambda state: state.object_metadata.is_sdc == False
    )

    # our model actor for sdc
    actor_ego = EgoAgent(
        model,
        is_controlled_func=lambda state: state.object_metadata.is_sdc == True
    )

    actors = [actor_ego, IDM_actors]
    select_action_list = [actor.select_action for actor in actors]

    states = [env.reset(scenario)]

    if steps is None:
        steps = states[0].remaining_timesteps

    for timestep in range(0, steps):
        current_state = states[-1]
        outputs = [
            select_action({'timestep': timestep}, current_state, None, None)
            for select_action in select_action_list
        ]
        # make action for all objects on scene
        action = agents.merge_actions(outputs)

        # make step
        next_state = env.step(current_state, action)
        states.append(next_state)

    return states[1:]

In [23]:
from collections import defaultdict
from waymax import metrics

def plot_states(states, use_log_traj=False, batch_idx=0):
    imgs = []
    for state in states:
        imgs.append(visualization.plot_simulator_state(
            state, use_log_traj=use_log_traj, batch_idx=batch_idx))
    mediapy.show_video(imgs, fps=10)

def get_ego_metrics(states, start_index=0):
    metrics_config = _config.MetricsConfig()

    metrics_per_time = defaultdict(list)
    for state in states[start_index:]:
        all_metrics = metrics.run_metrics(state, metrics_config)
        for k, v in all_metrics.items():
            metrics_per_time[k].append(np.asarray(v.value[state.object_metadata.is_sdc]))

    return {
        k: np.mean(metrics_per_time[k], axis=0)
        for k in metrics_per_time.keys()
    }

In [24]:
scenario = next(iter(val_dataset))

In [25]:
states = generate_close_loop(model, scenario, steps=50)
print(get_ego_metrics(states))
plot_states(states, batch_idx=0)

{'log_divergence': array([0.17863446, 0.22353138, 0.18882988, 0.46869758], dtype=float32), 'overlap': array([0., 0., 0., 0.], dtype=float32), 'offroad': array([0., 0., 0., 0.], dtype=float32)}


This browser does not support the video tag.


In [26]:
plot_states(states, batch_idx=1)

This browser does not support the video tag.
